In [164]:
!pip install sympy

Defaulting to user installation because normal site-packages is not writeable


In [165]:
import sympy as sp
from IPython.display import display, Math


In [166]:
def ode_to_transfer_function(equation):
    """
    Automatically identifies the system type and converts
    a linear ODE into a transfer function G(s) = Y(s)/U(s)

    Rules:
    - y = output
    - u = input
    - **n = nth derivative
    - Poles are preserved (no simplification)
    """

    t, s = sp.symbols('t s')
    y = sp.Function('y')(t)
    u = sp.Function('u')(t)

    equation = equation.replace(" ", "")
    lhs_str, rhs_str = equation.split("=")

    # --- system type detection ---
    if "M" in equation:
        system_type = "Mechanical Translational System"
    elif "J" in equation:
        system_type = "Mechanical Rotational System"
    else:
        system_type = "General Linear System"

    # --- parse derivatives ---
    def parse(expr):
        expr = expr.replace("y**3", "D3y")
        expr = expr.replace("y**2", "D2y")
        expr = expr.replace("y**1", "D1y")
        expr = expr.replace("u**3", "D3u")
        expr = expr.replace("u**2", "D2u")
        expr = expr.replace("u**1", "D1u")

        return sp.sympify(expr, locals={
            "D3y": y.diff(t, 3),
            "D2y": y.diff(t, 2),
            "D1y": y.diff(t, 1),
            "y": y,
            "D3u": u.diff(t, 3),
            "D2u": u.diff(t, 2),
            "D1u": u.diff(t, 1),
            "u": u
        })

    lhs = parse(lhs_str)
    rhs = parse(rhs_str)

    Y, U = sp.symbols('Y U')

    laplace = {
        y: Y,
        y.diff(t, 1): s*Y,
        y.diff(t, 2): s**2*Y,
        u: U,
        u.diff(t, 1): s*U,
        u.diff(t, 2): s**2*U
    }

    eq_L = lhs.subs(laplace) - rhs.subs(laplace)
    Y_sol = sp.solve(eq_L, Y, simplify=False)[0]
    Gs = Y_sol / U

    # force polynomial denominator
    num, den = sp.fraction(Gs)
    Gs = num / sp.expand(den)

    # show detected system
    display(Math(r"\textbf{Detected System: " + system_type + "}"))

    return Gs

In [167]:
equation = "J*y**2 + B*y**1 = K*u"

In [168]:
Gs = ode_to_transfer_function(equation)

<IPython.core.display.Math object>

In [169]:
display(
    Math(
        r"G(s) = \frac{Y(s)}{U(s)} = " + sp.latex(Gs)
    )
)

<IPython.core.display.Math object>